In [13]:
import pandas as pd
import seaborn as sns
import numpy as np
from collections import Counter

In [14]:
df = pd.read_csv('fake_job_postings.csv')
rng = np.random.RandomState(20201024)
test = df.sample(frac=0.20, random_state=rng)
train_mask = pd.Series(True, index=df.index)
train_mask[test.index] = False
train = df[train_mask].copy()

In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 14304 entries, 0 to 17878
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               14304 non-null  int64 
 1   title                14304 non-null  object
 2   location             14027 non-null  object
 3   department           5095 non-null   object
 4   salary_range         2291 non-null   object
 5   company_profile      11614 non-null  object
 6   description          14304 non-null  object
 7   requirements         12142 non-null  object
 8   benefits             8538 non-null   object
 9   telecommuting        14304 non-null  int64 
 10  has_company_logo     14304 non-null  int64 
 11  has_questions        14304 non-null  int64 
 12  employment_type      11495 non-null  object
 13  required_experience  8637 non-null   object
 14  required_education   7787 non-null   object
 15  industry             10355 non-null  object
 16  func

In [16]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

In [17]:
train['company_profile'] = train['company_profile'].fillna('')

In [18]:
train['comp_info'] = train['company_profile'] + train['description']

In [19]:
bayes_pipe = Pipeline([
    ('word_count', CountVectorizer()),
    ('classify', MultinomialNB())
])
bayes_pipe.fit(train['comp_info'], train['fraudulent'])

Pipeline(steps=[('word_count', CountVectorizer()),
                ('classify', MultinomialNB())])

In [20]:
train_d = bayes_pipe.predict(train['comp_info'])
bayes_pipe_acc_train = accuracy_score(train['fraudulent'], train_d)
bayes_pipe_acc_train

0.9659535794183445

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
cluster_pipe_2 = Pipeline([
    ('vectorize', TfidfVectorizer(stop_words='english', max_features=10000)),
    ('cluster', KMeans(2))
])

In [ ]:
cluster_pipe_2.fit(train['description'])
article_clusters = cluster_pipe_2.predict(train['description'])
article_df = pd.DataFrame(article_clusters)

In [ ]:
tech_word_df = pd.DataFrame(cluster_pipe_2['cluster'].cluster_centers_[1,:])
words = cluster_pipe_2['vectorize'].get_feature_names()
tech_word_df[1] = words
tech_word_df.sort_values(by = 0, ascending = False).head(25)